In [22]:
# 2. Import Library
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("Library berhasil diimpor.")
print("Library digunakan untuk cleaning, tokenisasi, stopword removal, dan stemming.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...


Library berhasil diimpor.
Library digunakan untuk cleaning, tokenisasi, stopword removal, dan stemming.


[nltk_data]   Package stopwords is already up-to-date!


In [23]:
# 3. Load Dataset
df = pd.read_csv("../dataset.csv")
text_column = "content"

print("Jumlah data:", len(df))
print("Nama kolom:", df.columns.tolist())

df.head(3)

Jumlah data: 225002
Nama kolom: ['userName', 'content', 'score', 'at', 'appVersion']


,userName,content,score,at,appVersion
0,Yuga Edit,akun gopay saya di blok,1,2022-01-21 10:52:12,4.9.3
1,ff burik,Lambat sekali sekarang ini bosssku apk gojek g...,3,2021-11-30 15:40:38,4.9.3
2,Anisa Suci Rahmayuliani,Kenapa sih dari kemarin sy buka aplikasi gojek...,4,2021-11-29 22:58:12,4.9.3


In [24]:
# 4. Case Folding
# Mengubah seluruh teks menjadi huruf kecil (lowercase):
def case_folding(text):
    return str(text).lower()

# Contoh
text = "Saya Belajar NLP"
print(case_folding(text))
# Output: saya belajar nlp



saya belajar nlp


In [25]:
# 5. Menghapus URL
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Contoh
text = "Kunjungi https://example.com sekarang"
print(remove_url(text))
# Output: Kunjungi  sekarang

Kunjungi  sekarang


In [26]:
# 6. Menghapus Mention
def remove_mention(text):
    return re.sub(r"@\w+", "", text)

# Contoh
text = "@user1 produk ini bagus"
print(remove_mention(text))

 produk ini bagus


In [27]:
# 7. Menghapus Hashtag
def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

# Contoh
text = "Saya sedang belajar #NLP"
print(remove_hashtag(text))
# Output: Saya sedang belajar 

Saya sedang belajar 


In [28]:
# 8. Menghapus Angka
def remove_number(text):
    return re.sub(r"\d+", "", text)

# Contoh
text = "Saya memiliki 5 buah apel"
print(remove_number(text))
# Output: Saya memiliki  buah apel

Saya memiliki  buah apel


In [29]:
# 9. Menghapus Punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

# Contoh
text = "Saya sedang belajar NLP!"
print(remove_punctuation(text))
# Output: Saya sedang belajar NLP

Saya sedang belajar NLP


In [30]:
# 10. Membersihkan Whitespace
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# Contoh
text = "   Saya sedang belajar NLP   "
print(clean_whitespace(text))
# Output: Saya sedang belajar NLP

Saya sedang belajar NLP


In [31]:
# 11. Final Cleaning Lengkap
# Gabungkan semua fungsi menjadi satu pipeline:
def clean_text(text):
    text = str(text)
    text = text.lower()           # case folding
    text = remove_url(text)       # hapus URL
    text = remove_mention(text)   # hapus mention
    text = remove_hashtag(text)   # hapus hashtag
    text = remove_number(text)    # hapus angka
    text = remove_punctuation(text)  # hapus punctuation
    text = clean_whitespace(text) # bersihkan whitespace
    return text

# Menerapkan ke dataset:
df["clean_text"] = df[text_column].apply(clean_text)

df[[text_column, "clean_text"]].head(5)

,content,clean_text
0,akun gopay saya di blok,akun gopay saya di blok
1,Lambat sekali sekarang ini bosssku apk gojek g...,lambat sekali sekarang ini bosssku apk gojek g...
2,Kenapa sih dari kemarin sy buka aplikasi gojek...,kenapa sih dari kemarin sy buka aplikasi gojek...
3,Baru download gojek dan hape baru trus ditop u...,baru download gojek dan hape baru trus ditop u...
4,Mantap,mantap


In [32]:
# 12. Stopword Removal
# Menghapus kata-kata umum (stopwords) yang tidak memiliki makna penting dalam analisis teks. Kita akan menggunakan daftar stopwords bahasa Indonesia dari NLTK.
stop_words = set(stopwords.words("indonesian"))

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in stop_words
    ]

# Contoh
tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

['belajar', 'nlp', 'menarik']


In [33]:
# 13. Stemming dengan Sastrawi
# Stemming adalah proses mengubah kata ke bentuk dasarnya (root word). Kita akan menggunakan
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_token(tokens):
    return [stemmer.stem(token) for token in tokens]

# Contoh
text = "Mahasiswa sedang mempelajari pemrograman."
print(stemmer.stem(text))

mahasiswa sedang ajar pemrograman


In [55]:
# 14. Pipeline Preprocessing Lengkap
import time

def preprocessing_pipeline(text):
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_token(tokens)
    return tokens

print("Memulai preprocessing seluruh dataset...")
waktu_mulai = time.time()

# Pertahankan 100 data yang sudah selesai
for i in range(0, len(df), 1000):
    batch = df[text_column].iloc[i:i+1000]

    # Lewati data yang sudah diproses
    belum_diproses = df.loc[batch.index, "processed_tokens"].isna()
    batch = batch[belum_diproses]

    if len(batch) == 0:
        continue

    hasil_batch = batch.apply(preprocessing_pipeline)

    # Simpan langsung setiap batch ke DataFrame
    df.loc[hasil_batch.index, "processed_tokens"] = hasil_batch

    print(
        f"Data diperiksa sampai indeks {min(i + 1000, len(df))} "
        f"dari {len(df)}"
    )

waktu_selesai = time.time()

print("Preprocessing selesai!")
print("Data yang sudah diproses:",
      df["processed_tokens"].notna().sum())
print("Waktu proses:", round(waktu_selesai - waktu_mulai, 2), "detik")

Memulai preprocessing seluruh dataset...
Data diperiksa sampai indeks 21000 dari 225002
Data diperiksa sampai indeks 22000 dari 225002
Data diperiksa sampai indeks 23000 dari 225002
Data diperiksa sampai indeks 24000 dari 225002
Data diperiksa sampai indeks 25000 dari 225002
Data diperiksa sampai indeks 26000 dari 225002
Data diperiksa sampai indeks 27000 dari 225002
Data diperiksa sampai indeks 28000 dari 225002
Data diperiksa sampai indeks 29000 dari 225002
Data diperiksa sampai indeks 30000 dari 225002
Data diperiksa sampai indeks 31000 dari 225002
Data diperiksa sampai indeks 32000 dari 225002
Data diperiksa sampai indeks 33000 dari 225002
Data diperiksa sampai indeks 34000 dari 225002
Data diperiksa sampai indeks 35000 dari 225002
Data diperiksa sampai indeks 36000 dari 225002
Data diperiksa sampai indeks 37000 dari 225002
Data diperiksa sampai indeks 38000 dari 225002
Data diperiksa sampai indeks 39000 dari 225002
Data diperiksa sampai indeks 40000 dari 225002
Data diperiksa samp

In [58]:
# Simpan hasil preprocessing
df.to_csv("dataset_preprocessed.csv", index=False)

print("Hasil preprocessing berhasil disimpan!")
print("Jumlah baris:", len(df))
print("Processed text terisi:", df["processed_text"].notna().sum())

Hasil preprocessing berhasil disimpan!
Jumlah baris: 225002
Processed text terisi: 225002


In [59]:
# Periksa jumlah hasil preprocessing di memori
print("Jumlah baris:", len(df))
print("Processed text terisi:", df["processed_text"].notna().sum())

# Simpan hasil preprocessing ke lokasi yang pasti
df.to_csv(
    r"D:\Projects\Natural Language Processing\pertemuan-03\dataset_preprocessed.csv",
    index=False
)

print("Dataset berhasil disimpan.")

Jumlah baris: 225002
Processed text terisi: 225002
Dataset berhasil disimpan.


In [60]:
# Pastikan df di memori masih memiliki hasil preprocessing
print("Jumlah hasil preprocessing:", df["processed_text"].notna().sum())

# Simpan hanya hasil preprocessing sebagai file cadangan
df[["processed_text"]].to_csv(
    r"D:\Projects\Natural Language Processing\pertemuan-03\processed_text_backup.csv",
    index=False
)

print("File cadangan berhasil disimpan.")

Jumlah hasil preprocessing: 225002
File cadangan berhasil disimpan.


In [39]:
# 15. Menggabungkan Token Kembali
df["processed_text"] = df["processed_tokens"].apply(
    lambda tokens: " ".join(tokens) if isinstance(tokens, list) else ""
)

print("Hasil penggabungan token menjadi teks:")
display(df[[text_column, "processed_tokens", "processed_text"]].head(5))

Hasil penggabungan token menjadi teks:


,content,processed_tokens,processed_text
0,akun gopay saya di blok,"[akun, gopay, blok]",akun gopay blok
1,Lambat sekali sekarang ini bosssku apk gojek g...,"[lambat, bosssku, apk, gojek, gk, kaya]",lambat bosssku apk gojek gk kaya
2,Kenapa sih dari kemarin sy buka aplikasi gojek...,"[sih, kemarin, sy, buka, aplikasi, gojek, kasi...",sih kemarin sy buka aplikasi gojek kasih binta...
3,Baru download gojek dan hape baru trus ditop u...,"[download, gojek, hape, trus, top, u, gopay, t...",download gojek hape trus top u gopay transaksi...
4,Mantap,[mantap],mantap


In [41]:
# 16. Perbandingan Raw vs Processed
for i in range(3):
    print("RAW:")
    print(df.iloc[i][text_column])

    print("\nPROCESSED:")
    print(df.iloc[i]["processed_text"])

    print("=" * 80)

RAW:
akun gopay saya di blok

PROCESSED:
akun gopay blok
RAW:
Lambat sekali sekarang ini bosssku apk gojek gk kaya dulu

PROCESSED:
lambat bosssku apk gojek gk kaya
RAW:
Kenapa sih dari kemarin sy buka aplikasi gojek malah keluar sendiri terus Saya kasih bintang 2 dulu kalo sudah normal sy kasih bintang 7

PROCESSED:
sih kemarin sy buka aplikasi gojek kasih bintang kalo normal sy kasih bintang


In [57]:
# 17. Menyimpan Dataset Hasil Preprocessing

output_columns = [text_column, "processed_text"]

df[output_columns].to_csv(
    "dataset_preprocessed.csv",
    index=False
)

print("Dataset berhasil disimpan!")
print("Lokasi: pertemuan-03/dataset_preprocessed.csv")
print("Jumlah data:", len(df))

Dataset berhasil disimpan!
Lokasi: pertemuan-03/dataset_preprocessed.csv
Jumlah data: 225002


In [44]:
# E. Tugas Eksperimen
# Perbandingan Tahapan Text Normalization pada 20 Data

def eksperimen_normalisasi(text):
    # 1. Case folding
    lowercase = case_folding(text)

    # 2. Cleaning
    cleaning = clean_text(lowercase)

    # 3. Tokenisasi
    tokenisasi = word_tokenize(cleaning)

    # 4. Stopword removal
    stopword = remove_stopwords(tokenisasi)

    # 5. Stemming
    stemming = stem_token(stopword)

    return pd.Series({
        "Raw Text": text,
        "Setelah Lowercase": lowercase,
        "Setelah Cleaning": cleaning,
        "Setelah Tokenisasi": str(tokenisasi),
        "Setelah Stopword": str(stopword),
        "Setelah Stemming": " ".join(stemming)
    })

print("Melakukan eksperimen pada 20 data pertama...")

tabel_eksperimen = df[text_column].dropna().head(20).apply(
    eksperimen_normalisasi
)

display(tabel_eksperimen)

print("Jumlah data eksperimen:", len(tabel_eksperimen))


Melakukan eksperimen pada 20 data pertama...


,Raw Text,Setelah Lowercase,Setelah Cleaning,Setelah Tokenisasi,Setelah Stopword,Setelah Stemming
0,akun gopay saya di blok,akun gopay saya di blok,akun gopay saya di blok,"['akun', 'gopay', 'saya', 'di', 'blok']","['akun', 'gopay', 'blok']",akun gopay blok
1,Lambat sekali sekarang ini bosssku apk gojek g...,lambat sekali sekarang ini bosssku apk gojek g...,lambat sekali sekarang ini bosssku apk gojek g...,"['lambat', 'sekali', 'sekarang', 'ini', 'bosss...","['lambat', 'bosssku', 'apk', 'gojek', 'gk', 'k...",lambat bosssku apk gojek gk kaya
2,Kenapa sih dari kemarin sy buka aplikasi gojek...,kenapa sih dari kemarin sy buka aplikasi gojek...,kenapa sih dari kemarin sy buka aplikasi gojek...,"['kenapa', 'sih', 'dari', 'kemarin', 'sy', 'bu...","['sih', 'kemarin', 'sy', 'buka', 'aplikasi', '...",sih kemarin sy buka aplikasi gojek kasih binta...
3,Baru download gojek dan hape baru trus ditop u...,baru download gojek dan hape baru trus ditop u...,baru download gojek dan hape baru trus ditop u...,"['baru', 'download', 'gojek', 'dan', 'hape', '...","['download', 'gojek', 'hape', 'trus', 'ditop',...",download gojek hape trus top u gopay transaksi...
4,Mantap,mantap,mantap,['mantap'],['mantap'],mantap
5,Bagus,bagus,bagus,['bagus'],['bagus'],bagus
6,Coba dulu,coba dulu,coba dulu,"['coba', 'dulu']",['coba'],coba
7,Ok,ok,ok,['ok'],['ok'],ok
8,Gimana ini kak pin saya salah terus padahal ud...,gimana ini kak pin saya salah terus padahal ud...,gimana ini kak pin saya salah terus padahal ud...,"['gimana', 'ini', 'kak', 'pin', 'saya', 'salah...","['gimana', 'kak', 'pin', 'salah', 'udah', 'uba...",gimana kak pin salah udah ubah salah
9,Biar aman kamu tidak bisa pakai gojek Jadi say...,biar aman kamu tidak bisa pakai gojek jadi say...,biar aman kamu tidak bisa pakai gojek jadi say...,"['biar', 'aman', 'kamu', 'tidak', 'bisa', 'pak...","['biar', 'aman', 'pakai', 'gojek', 'uninstal',...",biar aman pakai gojek uninstal biar aman


Jumlah data eksperimen: 20


In [ ]:
# F. Analisis Penting
1. Apakah semua stopword harus dihapus? Berikan contoh ketika stopword penting untuk dipertahankan.
Tidak semua stopword harus dihapus karena ada kata yang memengaruhi makna kalimat. Contohnya, kata “tidak” pada kalimat “aplikasi ini tidak bagus” harus dipertahankan karena menunjukkan bahwa ulasan tersebut bernilai negatif.
2. Apakah stemming selalu meningkatkan kualitas data? Kapan stemming dapat merugikan?
Tidak selalu, karena stemming bisa mengubah kata menjadi kata dasar dan menghilangkan informasi tertentu. Stemming dapat merugikan jika kata yang memiliki makna berbeda berubah menjadi bentuk yang sama.
3. Apakah angka harus selalu dihapus? Berikan contoh kasus di mana angka penting.
Tidak, karena angka bisa mengandung informasi penting. Contohnya, angka pada ulasan “Gojek mendapat rating 5 bintang” menunjukkan penilaian pengguna terhadap aplikasi.
4. Apakah URL selalu tidak berguna? Kapan URL dapat menjadi fitur yang informatif?
Tidak selalu. URL dapat berguna jika penelitian membahas tautan penipuan, situs tertentu, atau sumber informasi yang dibagikan pengguna.
5. Apakah hashtag harus selalu dihapus?
Tidak, karena hashtag bisa menunjukkan topik pembicaraan. Contohnya, #Gojek dan #Promo dapat membantu mengetahui topik yang dibahas dalam ulasan.
6. Apakah preprocessing yang sama cocok untuk semua task NLP?
Tidak, karena setiap task NLP memiliki tujuan yang berbeda. Misalnya, dalam analisis sentimen, kata “tidak” perlu dipertahankan karena memengaruhi makna sentimen, sedangkan pada task lain perlakuannya bisa berbeda.
Kesimpulan
Preprocessing perlu disesuaikan dengan karakteristik data dan tujuan NLP yang dilakukan. Tidak semua kata, angka, URL, atau hashtag harus dihapus karena sebagian bisa mengandung informasi penting untuk analisis.

In [54]:
print("Jumlah baris dataset:", len(df))
print("Indeks terakhir:", df.index[-1])
print("Jumlah processed_tokens terisi:",
      df["processed_tokens"].notna().sum())

Jumlah baris dataset: 225002
Indeks terakhir: 225001
Jumlah processed_tokens terisi: 20000


In [61]:
print("df masih ada:", "df" in globals())

if "df" in globals():
    print("Jumlah baris:", len(df))
    print("Hasil preprocessing terisi:", df["processed_text"].notna().sum())
    print(df["processed_text"].head(3).tolist())

df masih ada: True
Jumlah baris: 225002
Hasil preprocessing terisi: 225002
['akun gopay blok', 'lambat bosssku apk gojek gk kaya', 'sih kemarin sy buka aplikasi gojek kasih bintang kalo normal sy kasih bintang']


In [62]:
# Simpan hasil preprocessing dalam format pickle
df[["processed_text"]].to_pickle(
    r"D:\Projects\Natural Language Processing\pertemuan-03\processed_text_backup.pkl"
)

print("Backup berhasil disimpan:", len(df), "baris")

Backup berhasil disimpan: 225002 baris


In [63]:
import pandas as pd
import csv

file_csv_baru = r"D:\Projects\Natural Language Processing\pertemuan-03\dataset_preprocessed_baru.csv"

# Simpan ulang seluruh dataset dengan format CSV
df.to_csv(
    file_csv_baru,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL
)

print("CSV baru berhasil disimpan.")
print("Jumlah baris:", len(df))
print("Teks preprocessing terisi:", df["processed_text"].notna().sum())

CSV baru berhasil disimpan.
Jumlah baris: 225002
Teks preprocessing terisi: 225002


In [65]:
print("Jumlah baris:", len(df))
print("Jumlah teks terisi:", df["processed_text"].notna().sum())
print("Contoh teks:")
print(df["processed_text"].head(10).tolist())

Jumlah baris: 225002
Jumlah teks terisi: 225002
Contoh teks:
['akun gopay blok', 'lambat bosssku apk gojek gk kaya', 'sih kemarin sy buka aplikasi gojek kasih bintang kalo normal sy kasih bintang', 'download gojek hape trus top u gopay transaksi dialfamart transaksi bloked transaksilaporan pusat bantu gak lainlama gojek kelola tokopedia udah nyimpangapa gojek anak bangsa tokopedia', 'mantap', '', '', '', '', '']


In [66]:
for kolom in ["content", "clean_text", "processed_tokens", "processed_text"]:
    if kolom in df.columns:
        teks = df[kolom].fillna("").astype(str).str.strip()
        print(f"{kolom}: {teks.ne('').sum()} data terisi")

content: 224923 data terisi
clean_text: 224969 data terisi
processed_tokens: 225002 data terisi
processed_text: 5 data terisi


In [67]:
print(df["processed_tokens"].head(10).tolist())
print("\nTipe data:")
print(df["processed_tokens"].apply(type).value_counts())

[['akun', 'gopay', 'blok'], ['lambat', 'bosssku', 'apk', 'gojek', 'gk', 'kaya'], ['sih', 'kemarin', 'sy', 'buka', 'aplikasi', 'gojek', 'kasih', 'bintang', 'kalo', 'normal', 'sy', 'kasih', 'bintang'], ['download', 'gojek', 'hape', 'trus', 'top', 'u', 'gopay', 'transaksi', 'dialfamart', 'transaksi', 'bloked', 'transaksilaporan', 'pusat', 'bantu', 'gak', 'lainlama', 'gojek', 'kelola', 'tokopedia', 'udah', 'nyimpangapa', 'gojek', 'anak', 'bangsa', 'tokopedia'], ['mantap'], ['bagus'], ['coba'], ['ok'], ['gimana', 'kak', 'pin', 'salah', 'udah', 'ubah', 'salah'], ['biar', 'aman', 'pakai', 'gojek', 'uninstal', 'biar', 'aman']]

Tipe data:
processed_tokens
<class 'list'>    225002
Name: count, dtype: int64


In [68]:
# Memperbaiki processed_text dari processed_tokens
df["processed_text"] = df["processed_tokens"].apply(
    lambda tokens: " ".join(tokens) if isinstance(tokens, list) else ""
)

# Periksa hasilnya
print("Jumlah baris:", len(df))
print(
    "Teks terisi:",
    df["processed_text"].fillna("").str.strip().ne("").sum()
)
print("\nContoh teks:")
print(df["processed_text"].head(10).tolist())

Jumlah baris: 225002
Teks terisi: 221180

Contoh teks:
['akun gopay blok', 'lambat bosssku apk gojek gk kaya', 'sih kemarin sy buka aplikasi gojek kasih bintang kalo normal sy kasih bintang', 'download gojek hape trus top u gopay transaksi dialfamart transaksi bloked transaksilaporan pusat bantu gak lainlama gojek kelola tokopedia udah nyimpangapa gojek anak bangsa tokopedia', 'mantap', 'bagus', 'coba', 'ok', 'gimana kak pin salah udah ubah salah', 'biar aman pakai gojek uninstal biar aman']


In [69]:
import csv

file_csv = r"D:\Projects\Natural Language Processing\pertemuan-03\dataset_preprocessed_final.csv"

df.to_csv(
    file_csv,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL
)

print("CSV final berhasil disimpan.")

CSV final berhasil disimpan.


In [70]:
cek = pd.read_csv(file_csv, encoding="utf-8", low_memory=False)

print("Jumlah baris:", len(cek))
print(
    "Teks terisi:",
    cek["processed_text"].fillna("").str.strip().ne("").sum()
)
print(cek["processed_text"].head(5).tolist())

Jumlah baris: 225002
Teks terisi: 221099
['akun gopay blok', 'lambat bosssku apk gojek gk kaya', 'sih kemarin sy buka aplikasi gojek kasih bintang kalo normal sy kasih bintang', 'download gojek hape trus top u gopay transaksi dialfamart transaksi bloked transaksilaporan pusat bantu gak lainlama gojek kelola tokopedia udah nyimpangapa gojek anak bangsa tokopedia', 'mantap']


In [71]:
# Bandingkan teks sebelum dan sesudah disimpan
sebelum = df["processed_text"].fillna("").astype(str).str.strip()
sesudah = cek["processed_text"].fillna("").astype(str).str.strip()

print("Terisi sebelum:", sebelum.ne("").sum())
print("Terisi sesudah:", sesudah.ne("").sum())
print("Baris yang berbeda:", (sebelum != sesudah).sum())

# Tampilkan maksimal 10 baris yang berbeda
beda = df.loc[sebelum != sesudah, ["processed_text"]].copy()
beda["hasil_csv"] = cek.loc[sebelum != sesudah, "processed_text"]

display(beda.head(10))

Terisi sebelum: 221180
Terisi sesudah: 221099
Baris yang berbeda: 81


,processed_text,hasil_csv
1163,nan,NaN
5650,nan,NaN
10929,nan,NaN
13218,nan,NaN
15359,nan,NaN
20053,nan,NaN
29397,nan,NaN
29941,nan,NaN
30825,nan,NaN
43988,nan,NaN


In [72]:
beda_idx = df.index[sebelum != sesudah][:10]

for i in beda_idx:
    print(f"\nBaris index: {i}")
    print("Sebelum:", repr(sebelum.loc[i]))
    print("Sesudah:", repr(sesudah.loc[i]))


Baris index: 1163
Sebelum: 'nan'
Sesudah: ''

Baris index: 5650
Sebelum: 'nan'
Sesudah: ''

Baris index: 10929
Sebelum: 'nan'
Sesudah: ''

Baris index: 13218
Sebelum: 'nan'
Sesudah: ''

Baris index: 15359
Sebelum: 'nan'
Sesudah: ''

Baris index: 20053
Sebelum: 'nan'
Sesudah: ''

Baris index: 29397
Sebelum: 'nan'
Sesudah: ''

Baris index: 29941
Sebelum: 'nan'
Sesudah: ''

Baris index: 30825
Sebelum: 'nan'
Sesudah: ''

Baris index: 43988
Sebelum: 'nan'
Sesudah: ''


In [76]:
sebelum_bersih = (
    df["processed_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace(["nan", "null"], "")
)

cek_bersih = (
    cek["processed_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace(["nan", "null"], "")
)

print("Teks terisi sebelum:", sebelum_bersih.ne("").sum())
print("Teks terisi sesudah:", cek_bersih.ne("").sum())

Teks terisi sebelum: 221099
Teks terisi sesudah: 221099
